# Step 5: Route Profiling & Unsupervised Clustering (KMeans)

## Objectives
1. Construct route-level profile features using demand and matched service productivity.
2. Scale features and perform KMeans clustering to segment CTA bus routes.
3. Inspect cluster centers and assign clear, explainable labels for planner analysis.
4. Save route profiles and cluster labels for backend API integration.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

PROCESSED_DATA_PATH = os.path.join("..", "data", "processed")

# Load 2-year demand features and matched demand-service dataset
df_demand = pd.read_parquet(os.path.join(PROCESSED_DATA_PATH, "demand_features_2yr.parquet"))
df_matched = pd.read_parquet(os.path.join(PROCESSED_DATA_PATH, "matched_demand_service.parquet"))

print(f"Loaded Demand Rows: {len(df_demand):,} | Matched Schedule Rows: {len(df_matched):,}")

Loaded Demand Rows: 83,997 | Matched Schedule Rows: 28,374


In [2]:
# 1. Aggregate Demand Profile per Route
demand_profile = df_demand.groupby('route').agg(
    avg_daily_rides=('rides', 'mean'),
    std_daily_rides=('rides', 'std'),
    total_observed_days=('date', 'count')
).reset_index()

# 2. Compute Weekend vs. Weekday demand ratio
weekday_rides = df_demand[df_demand['is_weekend'] == 0].groupby('route')['rides'].mean()
weekend_rides = df_demand[df_demand['is_weekend'] == 1].groupby('route')['rides'].mean()

ratio_df = pd.DataFrame({
    'route': weekday_rides.index,
    'weekend_weekday_ratio': (weekend_rides / weekday_rides).fillna(0)
}).reset_index(drop=True)

# 3. Aggregate Service Productivity Profile per Route (from matched data)
service_profile = df_matched.groupby('route').agg(
    avg_trips_per_day=('scheduled_trips', 'mean'),
    avg_service_hours=('scheduled_service_hours', 'mean'),
    boardings_per_trip=('boardings_per_scheduled_trip', 'mean'),
    boardings_per_hour=('boardings_per_scheduled_service_hour', 'mean')
).reset_index()

# Merge into single Route Profile
route_profile = pd.merge(demand_profile, ratio_df, on='route', how='left')
route_profile = pd.merge(route_profile, service_profile, on='route', how='left').fillna(0)

print(f"Generated Profile for {len(route_profile)} unique routes.")
route_profile.head()

Generated Profile for 129 unique routes.


,route,avg_daily_rides,std_daily_rides,total_observed_days,weekend_weekday_ratio,avg_trips_per_day,avg_service_hours,boardings_per_trip,boardings_per_hour
0,1,1110.341176,254.376068,510,0.000000,59.000000,32.981486,19.103051,34.192085
1,10,574.374384,331.874182,203,1.168142,36.000000,26.544545,26.199495,35.590242
2,100,405.176471,187.042288,510,0.000000,52.777143,25.121600,7.085128,14.884149
3,1001,561.027548,1665.111559,726,4.055617,0.000000,0.000000,0.000000,0.000000
4,103,1304.150685,426.187935,730,0.505638,122.203187,57.292749,9.868799,21.009657


In [3]:
# Select numeric features for clustering
cluster_features = [
    'avg_daily_rides', 
    'std_daily_rides', 
    'weekend_weekday_ratio', 
    'avg_trips_per_day', 
    'boardings_per_trip'
]

X_cluster = route_profile[cluster_features]

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Fit KMeans (K=4 groups)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
route_profile['cluster'] = kmeans.fit_predict(X_scaled)

# Display Cluster Summary Profiles
cluster_summary = route_profile.groupby('cluster')[cluster_features].mean().round(2)
cluster_summary['route_count'] = route_profile['cluster'].value_counts()

print("=================== ROUTE CLUSTER PROFILES ===================")
print(cluster_summary)
print("==============================================================")

# Save cluster profile to processed data
cluster_output_path = os.path.join(PROCESSED_DATA_PATH, "route_clusters.parquet")
route_profile.to_parquet(cluster_output_path, index=False)
print(f"\nRoute cluster profiles saved to: {cluster_output_path}")

=================== ROUTE CLUSTER PROFILES ===================
         avg_daily_rides  std_daily_rides  weekend_weekday_ratio  \
cluster                                                            
0                1234.64           377.83                   0.37   
1               12799.52          2737.48                   0.72   
2                5994.46          1413.72                   0.59   
3                 663.08          1054.55                   3.10   

         avg_trips_per_day  boardings_per_trip  route_count  
cluster                                                      
0                    78.75               15.53           66  
1                   284.18               47.26           16  
2                   180.28               33.37           45  
3                     0.00                0.00            2  

Route cluster profiles saved to: ..\data\processed\route_clusters.parquet


## Summary: Route Profiling & Unsupervised Clustering

### Why Clustering is Essential for RouteRanker
In transit network planning, raw daily boarding counts alone do not reveal whether a bus route is performing efficiently relative to its scheduled supply. A low boarding number might indicate an underperforming line or simply a low-frequency coverage service meant for local access.

By running **KMeans Clustering ($K=4$)** on normalized demand features (`avg_daily_rides`, `std_daily_rides`, `weekend_weekday_ratio`) alongside operational supply metrics (`avg_trips_per_day`, `boardings_per_trip`), we segment the 129 active CTA bus routes into four distinct, explainable operational profiles:

---

### Cluster Breakdown & Operational Interpretation

1. **Cluster 0: Low-Volume Feeder / Local Coverage Routes** *(66 routes)*
   * **Key Metrics**: ~1,235 average daily rides, 78.8 daily trips, 15.5 boardings per trip.
   * **Significance**: These routes represent neighbourhood feeders, suburban links, or coverage-oriented services. They have low passenger turnover per trip but fulfill social equity and geographic access mandates. Planners should monitor these for coverage obligations rather than pure revenue productivity.

2. **Cluster 1: High-Density Commuter Trunk Lines** *(16 routes)*
   * **Key Metrics**: ~12,800 average daily rides, 284.2 daily trips, 47.3 boardings per trip, high weekend stability (0.72 weekend ratio).
   * **Significance**: These are CTA's core backbone lines carrying heavy all-day transit demand. High boardings per scheduled trip highlight candidate corridors for capacity expansion, articulated buses, or dedicated bus lanes.

3. **Cluster 2: Moderate Urban Connectors** *(45 routes)*
   * **Key Metrics**: ~5,994 average daily rides, 180.3 daily trips, 33.4 boardings per trip.
   * **Significance**: Secondary arterials with balanced demand and steady passenger throughput. These serve as primary candidates for schedule adjustments or minor frequency reallocations based on model forecasts.

4. **Cluster 3: Outlier / Special Event / Non-Standard Services** *(2 routes)*
   * **Key Metrics**: ~663 average daily rides, high weekend-to-weekday ratio (3.10), 0.0 scheduled daily trips in matched historical feeds.
   * **Significance**: Represents non-standard, express, or seasonal weekend-heavy routes (e.g., stadium shuttles, museum services) that lack regular scheduled daily templates in standard archives. Separating these ensures they do not skew regression models or standard operational review queues.

---

### Integration with RouteRanker Dashboard & API
The saved `route_clusters.parquet` dataset provides the core structural profile layer for the application backend. Planners can filter and compare peer routes within the same cluster rather than evaluating a low-volume neighborhood line against a high-density downtown corridor.